In [1]:
import os
import json
import time
from pathlib import Path
from dotenv import load_dotenv

In [2]:
from openai import OpenAI

In [3]:
import re

In [23]:
import math

In [68]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

True

In [69]:
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [15]:
PROCESSED_DIR = Path("processed")
OUTPUT_DIR    = Path("guidance_extracted")
FINANCIAL_DIR = Path("../data/financial_data")
OUTPUT_DIR.mkdir(exist_ok=True)


In [13]:
MEASURABLE_METRICS = [
    # Direct income statement fields
    "revenue", "total revenue", "gross profit", "operating income",
    "net income", "r&d", "research and development", "operating expenses",
    "ebit", "ebitda", "tax expense", "income tax",
    # Derived metrics
    "gross margin", "operating margin", "net margin", "net income margin",
    "profit margin", "Pre-tax Margin",
    # From earnings history
    "eps", "earnings per share",
    # Common aliases
    "top line", "bottom line", "sales"
]

print("Setup complete")
print(f"Measurable metrics defined: {len(MEASURABLE_METRICS)}")

Setup complete
Measurable metrics defined: 23


In [14]:
def contains_measurable_metric(sentence):
    sentence = sentence.lower()
    for metric in MEASURABLE_METRICS:
        pattern = r"\b" + re.escape(metric.lower()) + r"\b"
        if re.search(pattern, sentence):
            return True

    return False

In [15]:
def prefilter_guidance_sentences(guidance_sentences):
    """
    From all extracted guidance sentences, keep only those
    that mention at least one measurable metric.
    """
    filtered = []
    removed  = []

    for sentence in guidance_sentences:
        if contains_measurable_metric(sentence):
            filtered.append(sentence)
        else:
            removed.append(sentence)

    return filtered, removed

In [40]:
test_path = PROCESSED_DIR / "IBM_Q2_2023.json"
with open(test_path, encoding="utf-8") as f:
    test_data = json.load(f)

original  = test_data.get("forward_guidance", [])
filtered, removed = prefilter_guidance_sentences(original)

print(f"Original guidance sentences: {len(original)}")
print(f"After metric filter:         {len(filtered)}")
print(f"Removed (no measurable metric): {len(removed)}")
print()
print("KEPT:")
for s in filtered:
    print(f"  - {s[:200]}")
print()
print("REMOVED:")
for s in removed:
    print(f"  - {s[:200]}")

Original guidance sentences: 3
After metric filter:         2
Removed (no measurable metric): 1

KEPT:
  - We see constant-currency revenue growth of 3% to 5% and we expect free cash flow of about $10.5 billion, which I'll remind you is up over $1 billion year-to-year
  - We expect IBM's operating pre-tax margin to expand by about 0.5 point year-to-year, driven by a combination of product mix and progress on our productivity initiatives

REMOVED:
  - Later this year, we expect to close the acquisition of Apptio, which complements and advances our IT automation capabilities


In [41]:
def infer_next_quarter(current_quarter, current_year):
    """Returns (quarter, year) for next quarter."""
    if current_quarter == 4:
        return 1, current_year + 1
    return current_quarter + 1, current_year


def infer_full_year_quarter(current_year):
    """Full year guidance targets Q4 of that fiscal year."""
    return 4, current_year


def quarter_inference_context(transcript_quarter, transcript_year):
    """
    Build a clear context string for the LLM explaining
    how to infer target quarters from this specific transcript.
    """
    next_q, next_y = infer_next_quarter(transcript_quarter, transcript_year)

    return f"""
QUARTER INFERENCE RULES for this transcript (Q{transcript_quarter} {transcript_year}):
- "next quarter" = Q{next_q} {next_y}
- "next fiscal quarter" = Q{next_q} {next_y}
- "this quarter" or "current quarter" = Q{transcript_quarter} {transcript_year}
- "full year {transcript_year}" or "fiscal {transcript_year}" = Q4 {transcript_year}
- "FY{transcript_year}" = Q4 {transcript_year}
- "full year {transcript_year+1}" or "FY{transcript_year+1}" = Q4 {transcript_year+1}
- "second half" of {transcript_year} = Q4 {transcript_year}
- "Q1 {transcript_year+1}" = Q1 {transcript_year+1}
- If NO time period mentioned at all → assume Q{next_q} {next_y} (next quarter)
- NEVER leave target_quarter or target_year as null
"""

In [69]:
def extract_structured_guidance(transcript_data):
    """
    Takes preprocessed transcript data.
    Returns list of structured guidance claims with:
    - Only measurable metrics
    - Explicit target_quarter and target_year always filled
    """
    symbol = transcript_data["symbol"]
    quarter = transcript_data["quarter"]
    year = transcript_data["year"]

    raw_guidance = transcript_data.get("forward_guidance", [])

    if not raw_guidance:
        return []

    filtered_sentences, _ = prefilter_guidance_sentences(raw_guidance)

    if not filtered_sentences:
        return []

    sentences_text = "\n".join(
        f"{i+1}. {s}" for i, s in enumerate(filtered_sentences)
    )

    quarter_context = quarter_inference_context(quarter, year)

    prompt = f"""You are a financial analyst extracting verifiable forward guidance from an earnings call transcript.
IMPORTANT: Only extract guidance that is seems a future claim, not a past report. Each sentence should be about future. Otherwise skip it. 
Company: {symbol}
Transcript Quarter: Q{quarter} {year}

{quarter_context}

Guidance sentences to analyze:
{sentences_text}

For each valid guidance claim, return a JSON object with exactly these fields:
{{
  "metric": "standardized metric name from the sentence",
  "raw_value": "exact value as stated e.g. 43.5-44.5% or $90B or high single digits",
  "value_low": <lower bound as float, null if not applicable>,
  "value_high": <upper bound as float, null if not applicable>,
  "value_unit": "%" or "B" or "M" or "absolute",
  "target_quarter": <integer 1-4, NEVER null>,
  "target_year": <integer e.g. 2023, NEVER null>,
  "confidence": <0.0-1.0, how confident you are this is real measurable guidance>,
  "raw_sentence": "original sentence"
}}

Return ONLY a JSON array of valid claims. If no valid measurable guidance found, return [].
No other text."""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system",
             "content": "You are a strict financial analyst. Follow the instruction to find out genuine future guidance."
            },
            {"role": "user",
             "content": prompt
            }
        ],
        temperature=0.1
    )
    text = response.choices[0].message.content.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    try:
        result = json.loads(text)

        if not isinstance(result, list):
            return []

        cleaned = []
        next_q, next_y = infer_next_quarter(quarter, year)

        for item in result:
            cleaned.append(item)

        return cleaned

    except Exception as e:
        print(f"  Extraction error: {e}")
        return []


In [70]:
def process_all_transcripts():
    """
    Process all 592 transcripts.
    Saves one JSON file per company containing all 16 quarters
    of extracted guidance in order Q0-Q15.
    """
    all_files = sorted([
        f for f in PROCESSED_DIR.glob("*.json")
        if not f.name.startswith("_")
    ])

    print(f"Total transcripts to process: {len(all_files)}")
    print()

    # Group by company
    company_files = {}
    for filepath in all_files:
        symbol = filepath.stem.split("_")[0]
        if symbol not in company_files:
            company_files[symbol] = []
        company_files[symbol].append(filepath)

    print(f"Companies found: {len(company_files)}")
    print()

    for symbol, files in company_files.items():
        output_path = OUTPUT_DIR / f"{symbol}_guidance.json"

        if output_path.exists():
            print(f"{symbol}: Already processed, skipping")
            continue

        print(f"Processing {symbol} ({len(files)} quarters)...")
        company_guidance = []

        # Sort files chronologically
        sorted_files = sorted(files, key=lambda f: (
            int(f.stem.split("_")[2]),  # year
            int(f.stem.split("_")[1].replace("Q", ""))  # quarter
        ))

        for i, filepath in enumerate(sorted_files):
            with open(filepath, encoding="utf-8") as f:
                transcript_data = json.load(f)

            q = transcript_data["quarter"]
            y = transcript_data["year"]

            print(f"  [{i:2d}] Q{q} {y}...", end=" ")

            claims = extract_structured_guidance(transcript_data)

            company_guidance.append({
                "quarter_index": i,
                "quarter":       q,
                "year":          y,
                "symbol":        symbol,
                "claims_count":  len(claims),
                "claims":        claims
            })

            print(f"{len(claims)} claims")
            time.sleep(2.5)  # Groq rate limit

        # Save company file
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(company_guidance, f, indent=2, ensure_ascii=False)

        print(f"  Saved: {output_path.name}")
        print()


In [72]:
process_all_transcripts()

Total transcripts to process: 592

Companies found: 37

AAPL: Already processed, skipping
AFL: Already processed, skipping
ALL: Already processed, skipping
AMZN: Already processed, skipping
AON: Already processed, skipping
AXP: Already processed, skipping
BAC: Already processed, skipping
BK: Already processed, skipping
BLK: Already processed, skipping
C: Already processed, skipping
CB: Already processed, skipping
COF: Already processed, skipping
CRM: Already processed, skipping
CSCO: Already processed, skipping
GOOGL: Already processed, skipping
GS: Already processed, skipping
HIG: Already processed, skipping
IBM: Already processed, skipping
INTC: Already processed, skipping
JPM: Already processed, skipping
MET: Already processed, skipping
META: Already processed, skipping
MS: Already processed, skipping
MSFT: Already processed, skipping
NFLX: Already processed, skipping
NVDA: Already processed, skipping
ORCL: Already processed, skipping
PNC: Already processed, skipping
PRU: Already pr

In [74]:
total_companies = 0
total_quarters = 0
total_claims = 0

company_summary = []

for filepath in sorted(OUTPUT_DIR.glob("*_guidance.json")):
    with open(filepath, "r", encoding="utf-8") as f:
        company_data = json.load(f)

    total_companies += 1

    company_claims = 0

    for quarter in company_data:
        total_quarters += 1
        company_claims += quarter["claims_count"]

    total_claims += company_claims

    company_summary.append({
        "symbol": filepath.stem.replace("_guidance", ""),
        "quarters": len(company_data),
        "claims": company_claims
    })

print("=" * 60)
print(f"Companies Processed : {total_companies}")
print(f"Total Quarters      : {total_quarters}")
print(f"Total Guidance Claims: {total_claims}")
print("=" * 60)

print("\nPer Company Summary:")
for company in company_summary:
    print(f"{company['symbol']:6} | "
          f"{company['quarters']:2} quarters | "
          f"{company['claims']:3} claims")

Companies Processed : 37
Total Quarters      : 592
Total Guidance Claims: 1031

Per Company Summary:
AAPL   | 16 quarters |  45 claims
AFL    | 16 quarters |  20 claims
ALL    | 16 quarters |   0 claims
AMZN   | 16 quarters |   3 claims
AON    | 16 quarters |  58 claims
AXP    | 16 quarters |  66 claims
BAC    | 16 quarters |   9 claims
BK     | 16 quarters |  15 claims
BLK    | 16 quarters |   3 claims
C      | 16 quarters |  22 claims
CB     | 16 quarters |   2 claims
COF    | 16 quarters |   0 claims
CRM    | 16 quarters |  68 claims
CSCO   | 16 quarters |  60 claims
GOOGL  | 16 quarters |   6 claims
GS     | 16 quarters |   0 claims
HIG    | 16 quarters |   6 claims
IBM    | 16 quarters |  57 claims
INTC   | 16 quarters |  67 claims
JPM    | 16 quarters |   1 claims
MET    | 16 quarters |  13 claims
META   | 16 quarters |  29 claims
MS     | 16 quarters |   0 claims
MSFT   | 16 quarters | 238 claims
NFLX   | 16 quarters |  17 claims
NVDA   | 16 quarters |  25 claims
ORCL   | 16 qua

In [7]:
with open("../data/quarterly_sentiment.json", encoding="utf-8") as f:
    sentiment_list = json.load(f)

with open("../data/evasion_labels_usefullPair.json", encoding="utf-8") as f:
    evasion_pairs = json.load(f)

In [8]:
sentiment_lookup = {}
for entry in sentiment_list:
    key = f"{entry['symbol']}_{entry['quarter']}_{entry['year']}"
    sentiment_lookup[key] = entry

evasion_map = {}
for pair in evasion_pairs:
    key = f"{pair['symbol']}_{pair['quarter']}_{pair['year']}"
    if key not in evasion_map:
        evasion_map[key] = {"total": 0, "evasive": 0}
    evasion_map[key]["total"] += 1
    if pair["label"] == "EVASIVE":
        evasion_map[key]["evasive"] += 1


evasion_rate_lookup = {}
for key, counts in evasion_map.items():
    evasion_rate_lookup[key] = round(
        counts["evasive"] / max(counts["total"], 1), 4
    )

print("Sentiment records loaded:", len(sentiment_lookup))
print("Evasion rate records loaded:", len(evasion_rate_lookup))
print()

Sentiment records loaded: 592
Evasion rate records loaded: 592



In [11]:
# Counters
skipped_no_value    = 0
skipped_future_year = 0
skipped_no_fin_data = 0
skipped_unscorable  = 0
taken               = 0

raw_dataset = []

guidance_files = sorted(OUTPUT_DIR.glob("*_guidance.json"))

In [55]:
def get_financial_data_for_quarter(symbol, quarter, year):
    """
    Loads existing financial data file and finds the income statement
    entry for the specific target quarter/year.
    Returns the full income statement row as a dict, or None if not found.
    """
    fin_path = FINANCIAL_DIR / f"{symbol}_financials.json"

    if not fin_path.exists():
        return None

    with open(fin_path, encoding="utf-8") as f:
        fin_data = json.load(f)

    quarterly_income = fin_data.get("income_statement",[])#.get("quarterly_income", [])
    if not quarterly_income:
        return None
    for entry in quarterly_income:
        date = entry.get("fiscalDateEnding", "")
        if not date or len(date) < 7:
            continue
        entry_year    = int(date[:4])
        entry_month   = int(date[5:7])
        entry_quarter = (entry_month - 1) // 3 + 1

        if entry_year == year and entry_quarter == quarter:
            return entry

    return None


In [56]:
def get_earning_history_for_quarter(symbol, quarter, year):
    """
    Loads existing financial data file and finds the income statement
    entry for the specific target quarter/year.
    Returns the full income statement row as a dict, or None if not found.
    """
    fin_path = FINANCIAL_DIR / f"{symbol}_financials.json"

    if not fin_path.exists():
        return None

    with open(fin_path, encoding="utf-8") as f:
        fin_data = json.load(f)

    earning_history = fin_data.get("earnings_history",[])
    if not earning_history:
        return None
    for entry in earning_history:
        date = entry.get("fiscalDateEnding", "")
        if not date or len(date) < 7:
            continue
        entry_year    = int(date[:4])
        entry_month   = int(date[5:7])
        entry_quarter = (entry_month - 1) // 3 + 1

        if entry_year == year and entry_quarter == quarter:
            return entry

    return None

In [47]:
def score_fulfillment(claim, actual_result):
    """
    Compare extracted guidance claim against actual computed value.
    Returns fulfillment score 0.0 to 1.0.

    Scoring logic:
    - Actual >= guidance low end/guidance high end -> 1.0 (fully beat)
    - Actual < guidance low end -> abs(actual-guidance)/guidance then Score=e^(−5×RelativeError)
    """
    if not actual_result or actual_result.get("computed_value") is None:
        return None
    actual_value = float(actual_result["computed_value"])
    value_low    = claim.get("value_low")
    value_high   = claim.get("value_high")
    unit         = claim.get("value_unit", "%")
    direction    = claim.get("direction", "unknown")

    # Convert revenue/income from raw to billions if needed
    if unit == "B" and actual_value > 1000:
        actual_value = actual_value / 1_000_000_000
    elif unit == "M" and actual_value > 1000:
        actual_value = actual_value / 1_000_000

    # Case 1: specific range given (e.g. 43.5-44.5%)
    if value_low is not None:
        value_low  = float(value_low)
    if value_high is not None:
        value_high = float(value_high)
    if value_low is not None:
        if actual_value >= value_low:
                return 1.0  # beat end
        else:
            # below range — score based on how close
            miss_pct = abs(value_low - actual_value) / value_low
            score = math.exp(-5 * miss_pct)
            return score

    # Case 3: only direction given (increase/decrease/flat)
    elif direction == "increase":
        return 0.5
    elif direction == "flat":
        return 0.75
    else:
        return None

In [70]:
def compute_metric_from_financials(metric_name, financial_entry, earning_entry):
    """
    Given a metric name (e.g. 'Gross Margin') and a full income statement
    entry dict, use Groq to calculate the actual value of that metric.
    Returns dict with computed_value and unit.
    """
    if not financial_entry and not earning_entry:
        return None

    # Build a clean readable version of the financial data
    fin_text = "\n".join(
        f"  {k}: {v}"
        for k, v in financial_entry.items()
        if v not in (None, "None", "")
    )
    earning_text = "\n".join(
        f"  {k}: {v}"
        for k, v in earning_entry.items()
        if v not in (None, "None", "")
    )


    prompt = f"""You are a financial analyst. 
Given this income statement data for one quarter, calculate the value of: {metric_name}

Income Statement Data:
{fin_text}
Earnings History of this Quarter: 
{earning_text}
Instructions:
For each extracted guidance metric:
1.For Reveneu, all types of Reveneu should be calculated with standard formula of reveneu calculation.
2. If the metric exists directly in the income statement,
   return the corresponding value.
3. If the metric exists in earnings history,
   return the corresponding value.
4. If the metric can be derived mathematically from
   available income statement fields or earnings history
   compute it using standard financial formulas.
5. If the metric cannot be computed or found from the
   provided financial data and also from earning history return null.
6. If there is any fancy metric name don't get confused, just use standard formula to calculate it.
- All dollar values are in raw numbers (divide by 1B for billions)
Return ONLY this JSON, no other text:
{{
  "metric": "{metric_name}",
  "computed_value": <number or null>,
  "unit": "%" or "B" or "M" or "absolute",
  "explanation": "brief one line explanation of how you calculated it"
}}"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system",
             "content": "You are a strict financial analyst. Follow the instruction to calculate the desired metric from given financial data."
            },
            {"role": "user",
             "content": prompt
            }
        ],
        temperature=0.7
    )
    text = response.choices[0].message.content.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(text)
    except Exception as e:
        print(f"  Compute metric error: {e}")
        return None

In [19]:
PROCESSED_FILE = OUTPUT_DIR / "processed_claims.json"

if PROCESSED_FILE.exists():
    with open(PROCESSED_FILE, "r", encoding="utf-8") as f:
        processed_claims = json.load(f)
else:
    processed_claims = {}

In [71]:
for guidance_file in guidance_files:
    with open(guidance_file, encoding="utf-8") as f:
        company_guidance = json.load(f)

    for quarter_entry in company_guidance:
        symbol          = quarter_entry["symbol"]
        transcript_q    = quarter_entry["quarter"]
        transcript_y    = quarter_entry["year"]
        claims          = quarter_entry["claims"]

        # Sentiment and evasion for this transcript quarter
        sent_key     = f"{symbol}_{transcript_q}_{transcript_y}"
        sentiment    = sentiment_lookup.get(sent_key, {})
        evasion_rate = evasion_rate_lookup.get(sent_key, 0.3)

        for claim_idx, claim in enumerate(claims):

            claim_key = (f"{symbol}_{transcript_q}_{transcript_y}_{claim_idx}")
    
            # Already processed?
            if claim_key in processed_claims:
                print(f"  SKIP (already processed): {claim_key}")
                continue
            target_q = claim.get("target_quarter")
            target_y = claim.get("target_year")
            val_low  = claim.get("value_low")
            val_high = claim.get("value_high")

            # Skip 1: both value_low and value_high are None
            if val_low is None and val_high is None:
                skipped_no_value += 1
                processed_claims[claim_key] = True

                with open(PROCESSED_FILE, "w", encoding="utf-8") as f:
                    json.dump(processed_claims, f, indent=2)
                continue

            # Skip 2: target year is 2025 or beyond
            if target_y is None or target_y >= 2025:
                skipped_future_year += 1
                processed_claims[claim_key] = True

                with open(PROCESSED_FILE, "w", encoding="utf-8") as f:
                    json.dump(processed_claims, f, indent=2)
                continue

            # Fetch actual financial data for target quarter
            print(f"Getting fin_entry for {claim_idx}_{symbol}_{transcript_q}_{transcript_y}..." )
            fin_entry = get_financial_data_for_quarter(symbol, target_q, target_y)
            print("Done!")
            print(f"Getting earning_entry for {claim_idx}_{symbol}_{transcript_q}_{transcript_y}..." )
            earning_entry = get_earning_history_for_quarter(symbol, target_q, target_y)
            print("Done!")
            # Compute actual metric value
            print(f"Getting actual_result for {claim_idx}_{symbol}_{transcript_q}_{transcript_y}..." )
            actual_result = compute_metric_from_financials(claim["metric"], fin_entry, earning_entry)
            print("Done!")
            time.sleep(1.2)

            if not actual_result or actual_result.get("computed_value") is None:
                skipped_unscorable += 1
                processed_claims[claim_key] = True

                with open(PROCESSED_FILE, "w", encoding="utf-8") as f:
                    json.dump(processed_claims, f, indent=2)
                continue

            # Score fulfillment
            print(f"Getting score for {claim_idx}_{symbol}_{transcript_q}_{transcript_y}..." )
            score = score_fulfillment(claim, actual_result)
            print("Done!")
            if score is None:
                skipped_unscorable += 1
                processed_claims[claim_key] = True

                with open(PROCESSED_FILE, "w", encoding="utf-8") as f:
                    json.dump(processed_claims, f, indent=2)
                continue

            # Build one dataset row
            raw_dataset.append({
                # Identifiers (for debugging, not training features)
                "symbol":             symbol,
                "transcript_quarter": transcript_q,
                "transcript_year":    transcript_y,
                "target_quarter":     target_q,
                "target_year":        target_y,

                # Features — sentiment of transcript quarter
                "sentiment_positive": sentiment.get("positive", 0.0),
                "sentiment_negative": sentiment.get("negative", 0.0),
                "sentiment_neutral":  sentiment.get("neutral",  0.0),

                # Features — evasion rate of transcript quarter
                "evasion_rate": evasion_rate,

                # Features — guidance claim properties
                "metric":      claim["metric"],
                "value_low":   val_low,
                "value_high":  val_high,
                "value_unit":  claim.get("value_unit", "%"),
                "confidence":  claim.get("confidence", 0.8),

                # Actual result
                "actual_value": actual_result["computed_value"],
                "actual_unit":  actual_result["unit"],

                # Label
                "fulfillment_score": score
            })
            processed_claims[claim_key] = True

            with open(PROCESSED_FILE, "w", encoding="utf-8") as f:
                json.dump(processed_claims, f, indent=2)

            taken += 1
            print(f"  TAKEN: {symbol} Q{transcript_q} {transcript_y} → "
                  f"{claim['metric']} → Q{target_q} {target_y} → "
                  f"Actual: {actual_result['computed_value']} → Score: {score}")

print()
print("=" * 60)
print(f"TAKEN:                  {taken}")
print(f"Skipped (no value):     {skipped_no_value}")
print(f"Skipped (future year):  {skipped_future_year}")
print(f"Skipped (no fin data):  {skipped_no_fin_data}")
print(f"Skipped (unscorable):   {skipped_unscorable}")
print(f"Total raw dataset rows: {len(raw_dataset)}")


  SKIP (already processed): AAPL_1_2021_0
  SKIP (already processed): AAPL_1_2021_1
  SKIP (already processed): AAPL_2_2021_0
  SKIP (already processed): AAPL_2_2021_1
  SKIP (already processed): AAPL_3_2021_0
  SKIP (already processed): AAPL_3_2021_1
  SKIP (already processed): AAPL_4_2021_0
  SKIP (already processed): AAPL_4_2021_1
  SKIP (already processed): AAPL_4_2021_2
  SKIP (already processed): AAPL_1_2022_0
  SKIP (already processed): AAPL_1_2022_1
  SKIP (already processed): AAPL_1_2022_2
  SKIP (already processed): AAPL_2_2022_0
  SKIP (already processed): AAPL_3_2022_0
  SKIP (already processed): AAPL_3_2022_1
  SKIP (already processed): AAPL_4_2022_0
  SKIP (already processed): AAPL_4_2022_1
  SKIP (already processed): AAPL_1_2023_0
  SKIP (already processed): AAPL_1_2023_1
  SKIP (already processed): AAPL_1_2023_2
  SKIP (already processed): AAPL_1_2023_3
  SKIP (already processed): AAPL_1_2023_4
  SKIP (already processed): AAPL_2_2023_0
  SKIP (already processed): AAPL_3

In [73]:
# Save raw dataset immediately
output_path = Path("../data/guidance_fulfillment_raw.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(raw_dataset, f, indent=2)
print(f"Saved to: {output_path}")

Saved to: ..\data\guidance_fulfillment_raw.json


In [72]:
print(f"Total raw dataset rows: {len(raw_dataset)}")

Total raw dataset rows: 171


In [61]:
print(f"Skipped (unscorable):   {skipped_unscorable}")

Skipped (unscorable):   473
